In [1]:
import sys
sys.path.append("..")
import pandas as pd
from src.data import load_cuad, TARGET_CLAUSES
from src.split import split_contracts
from src.chunking import fixed_chunks, clause_chunks
from src.retrieval import make_dense_ranker
from src.metrics import evaluate_ranker

contracts = load_cuad()
dev_idx, test_idx = split_contracts(contracts)

for name, fn in [("fixed", fixed_chunks), ("clause", clause_chunks)]:
    sizes = [ch["end"] - ch["start"] for i in dev_idx for ch in fn(contracts[i]["text"])]
    s = pd.Series(sizes)
    print(name, "| chunks:", len(s), "| mean size:", round(s.mean()), "| median:", round(s.median()))

fixed | chunks: 23810 | mean size: 994 | median: 1000
clause | chunks: 18109 | mean size: 1106 | median: 928


In [2]:
ranker = make_dense_ranker(clause_chunks, tag="clause-200-2000-200")
quick = dev_idx[:40]
fixed_ranker = make_dense_ranker()

for cl in ["Governing Law", "Non-Compete"]:
    print(cl, "fixed ", evaluate_ranker(fixed_ranker, contracts, quick, cl))
    print(cl, "clause", evaluate_ranker(ranker, contracts, quick, cl))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Governing Law fixed  {'hit@1': 0.583, 'hit@3': 0.861, 'hit@5': 0.917, 'hit@10': 0.944, 'span_recall@1': 0.583, 'span_recall@3': 0.847, 'span_recall@5': 0.903, 'span_recall@10': 0.944, 'mrr': 0.731, 'n': 36}
Governing Law clause {'hit@1': 0.722, 'hit@3': 0.944, 'hit@5': 0.972, 'hit@10': 0.972, 'span_recall@1': 0.708, 'span_recall@3': 0.944, 'span_recall@5': 0.972, 'span_recall@10': 0.972, 'mrr': 0.833, 'n': 36}
Non-Compete fixed  {'hit@1': 0.333, 'hit@3': 0.583, 'hit@5': 0.667, 'hit@10': 0.833, 'span_recall@1': 0.333, 'span_recall@3': 0.465, 'span_recall@5': 0.514, 'span_recall@10': 0.708, 'mrr': 0.485, 'n': 12}
Non-Compete clause {'hit@1': 0.083, 'hit@3': 0.417, 'hit@5': 0.5, 'hit@10': 0.833, 'span_recall@1': 0.083, 'span_recall@3': 0.375, 'span_recall@5': 0.458, 'span_recall@10': 0.715, 'mrr': 0.305, 'n': 12}


In [3]:
rows = {cl: evaluate_ranker(ranker, contracts, dev_idx, cl) for cl in TARGET_CLAUSES}
clause_res = pd.DataFrame(rows).T
clause_res.to_csv("../eval/results_dense_clause_dev.csv")

fixed_res = pd.read_csv("../eval/results_dense_fixed1000_dev.csv", index_col=0)
delta = (clause_res - fixed_res)[["hit@1", "hit@3", "hit@5", "mrr"]].round(3)
delta

,hit@1,hit@3,hit@5,mrr
Governing Law,0.088,0.049,0.042,0.064
Renewal Term,0.102,0.059,0.042,0.079
Notice Period To Terminate Renewal,-0.056,-0.028,-0.014,-0.034
Termination For Convenience,0.034,-0.025,0.017,0.010
Non-Compete,-0.053,-0.014,-0.040,-0.041
Cap On Liability,0.100,0.016,0.000,0.056
Uncapped Liability,0.024,0.098,0.037,0.040
